# ALFRED DFII10 指定 2022 年底版本的公开图表 CSV 预检

单次只读 GET，保留原始字节和 NAS 回读；不读取 SOXX、拟合或跑账户。

In [1]:
import csv,hashlib,json,subprocess,tempfile
from datetime import datetime,timezone,date
from pathlib import Path
from src.data.nas_market_data import publish_nas_snapshot,materialize_nas_snapshot

root=Path.cwd();card=root/'research/stage1_treasury_real_yield_alfred_graph_vintage_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
raw=root/'data/snapshots/stage1_treasury_real_yield_alfred_graph_vintage_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','plan_sha256':sha(card/'plan.json'),'GET_attempts':1,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
policy=plan['request_policy'];body=raw/'asof_2022_12_30.body';headers=raw/'asof_2022_12_30.headers';started=datetime.now(timezone.utc).isoformat()
cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(policy['connect_timeout_seconds']),'--max-time',str(policy['total_timeout_seconds']),'--max-redirs',str(policy['redirects_max']),'--max-filesize',str(policy['max_response_bytes']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',policy['user_agent'],'-w','%{http_code} %{url_effective}',plan['exact_GET']]
try:
    proc=subprocess.run(cmd,capture_output=True,text=True,timeout=45);fields=proc.stdout.split(' ',1)
    receipt={'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'url':plan['exact_GET'],'curl_exit_code':proc.returncode,'http_status':int(fields[0]) if fields[0].isdigit() else None,'final_url':fields[1] if len(fields)>1 else None,'transport_error':proc.stderr[:250]}
except Exception as exc:
    receipt={'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'url':plan['exact_GET'],'transport_exception':type(exc).__name__+':'+str(exc)[:200]}
receipt.update(body_sha256=sha(body) if body.exists() else None,body_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
(raw/'capture_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
failure=None;parsed=None
if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure='transport_or_http'
else:
    try:
        with body.open(newline='',encoding='utf-8-sig') as f:reader=csv.DictReader(f);columns=reader.fieldnames;rows=list(reader)
        assert columns and 'DFII10' in [x.upper() for x in columns] and rows
        date_col=next(x for x in columns if x.lower() in ('date','observation_date'))
        value_col=next(x for x in columns if x.upper()=='DFII10')
        good=[]
        for row in rows:
            if row[value_col] in ('','.'):continue
            d=datetime.strptime(row[date_col],'%Y-%m-%d').date();v=float(row[value_col]);good.append((d,v))
        assert good and len(good)==len({d for d,_ in good})
        parsed={'columns':columns,'rows':len(rows),'finite_rows':len(good),'first_date':str(min(d for d,_ in good)),'last_date':str(max(d for d,_ in good)),'last_value':good[-1][1]}
        assert min(d for d,_ in good)<=date(2018,10,1) and max(d for d,_ in good)>=date(2022,12,28) and max(d for d,_ in good)<=date(2022,12,30)
    except Exception as exc:failure='csv_or_vintage_date_gate:'+type(exc).__name__+':'+str(exc)[:160]
manifest={'schema_version':1,'experiment':plan['experiment'],'plan_sha256':state['plan_sha256'],'receipt':receipt,'parsed':parsed,'failure':failure,'vintage_semantics_certified':False,'source_market_outcome_reads':0}
(raw/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n');(card/'source_receipt.json').write_text(json.dumps(manifest,indent=2)+'\n')
files={str(p.relative_to(root)):'raw' for p in raw.iterdir() if p.is_file()}
try:
    binding=publish_nas_snapshot(workspace_root=root,source_manifest=raw/'source_manifest.json',files=files,as_of=date.today().isoformat(),source_metadata={'experiment':plan['experiment'],'source':'ALFRED public graph CSV as-of probe'})
    (card/'nas_binding.json').write_text(json.dumps(binding,indent=2)+'\n')
    with tempfile.TemporaryDirectory(prefix='alfred-graph-cold-') as temp:
        readback=materialize_nas_snapshot(binding,workspace_root=Path(temp))
        for obj in binding['objects']:assert sha(Path(temp)/obj['local_path'])==obj['sha256']
    assert binding['verified'] and readback['restored_objects']==len(files)
except Exception as exc:
    failure=(failure+';' if failure else '')+'NAS:'+type(exc).__name__+':'+str(exc)[:180];binding=None
state.update(status='completed' if not failure else 'stopped',failure=failure);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
result={'decision':'ALFRED_asof_graph_CSV_transport_candidate_PIT_unqualified' if not failure else 'needs_ALFRED_graph_source_or_NAS','failure':failure,'GET_attempts':1,'http_status':receipt.get('http_status'),'parsed':parsed,'NAS_source_id':binding['source_id'] if binding else None,'NAS_readback_verified':bool(binding),'vintage_semantics_certified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n');print(json.dumps(result))


{"decision": "needs_ALFRED_graph_source_or_NAS", "failure": "transport_or_http", "GET_attempts": 1, "http_status": 0, "parsed": null, "NAS_source_id": "a26b04d6e9518173e1f27c652d73cbff54c5a2467a4409ac905ac7fb5753d3ee", "NAS_readback_verified": true, "vintage_semantics_certified": false, "SOXX_outcome_reads": 0, "model_fits": 0, "native_accounts": 0}


# ALFRED DFII10 历史图表导出：固定传输预检失败

只读请求固定为 `DFII10`、`vintage_date=2022-12-30` 的 ALFRED 公开图表 CSV。单次 GET 在 **35 秒**后以 `curl (28)` 超时，HTTP 状态为 0，响应头与数据体均为零字节。确切 URL、请求和结束 UTC、退出码以及空响应哈希见 `source_receipt.json`；接收清单已在 Yanxuan NAS 远端校验并定向恢复。没有重试，也没有更换版本日期。

这个结果只证明**当前执行环境无法经这条图表通道取得该指定版本**，不证明 ALFRED 没有历史数据。上一固定表单 POST 也超时，两条尝试均没有取得可比较的历史值。财政部回溯 CSV 仍不能代替逐日首次公开版本；因此实际收益率信息门、完整 Stage1 0bp Tiered 账户及 SOXX 硬门都没有运行或通过。

本卡 1 次 GET、0 SOXX 结果读取、0 拟合、0 账户。固定传输版本关闭，后续不得在同一复用历史上用改日期或重试次数救门。
